In [1]:
!pip install -q deepface opencv-python-headless tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.9 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
from deepface import DeepFace
from google.colab import files
from tqdm import tqdm

26-04-09 05:02:57 - Directory /root/.deepface has been created
26-04-09 05:02:57 - Directory /root/.deepface/weights has been created


In [3]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [4]:
print("Upload reference images of person")

uploaded = files.upload()

face_db = {}
person_id = "person_1"

face_db[person_id] = []

for filename in uploaded.keys():

    img = cv2.imread(filename)

    embedding = DeepFace.represent(
        img_path=img,
        model_name="ArcFace",      # HEAVY MODEL
        detector_backend="retinaface",
        enforce_detection=False
    )[0]["embedding"]

    face_db[person_id].append({
        "embedding": embedding,
        "type": "known_person"
    })

print("Face database created")

Upload reference images of person


Saving emp_name.jpeg to emp_name.jpeg
26-04-09 05:25:41 - 🔗 arcface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5 to /root/.deepface/weights/arcface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /root/.deepface/weights/arcface_weights.h5
100%|██████████| 137M/137M [00:09<00:00, 14.0MB/s]


26-04-09 05:25:55 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [01:17<00:00, 1.53MB/s]


Face database created


In [5]:
def detect_faces(frame, face_db):

    detections = []

    faces = DeepFace.extract_faces(
        img_path=frame,
        detector_backend="retinaface",
        enforce_detection=False
    )

    for face in faces:

        area = face["facial_area"]

        x = area["x"]
        y = area["y"]
        w = area["w"]
        h = area["h"]

        face_crop = frame[y:y+h, x:x+w]

        label = "unknown"
        person_id = None
        confidence = 0

        try:

            embedding = DeepFace.represent(
                img_path=face_crop,
                model_name="ArcFace",
                enforce_detection=False
            )[0]["embedding"]

            best_score = 0

            for pid, data_list in face_db.items():

                for data in data_list:

                    score = cosine_similarity(
                        embedding,
                        data["embedding"]
                    )

                    if score > best_score:

                        best_score = score
                        person_id = pid
                        label = data["type"]

            if best_score > 0.5:
                confidence = best_score

        except:
            pass

        detections.append({
            "label": label,
            "confidence": confidence,
            "bbox": [x, y, w, h]
        })

    return detections

In [6]:
print("Upload video")

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

Upload video


Saving test_cam1.mp4 to test_cam1.mp4


In [7]:
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output.mp4", fourcc, fps, (width, height))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

for _ in tqdm(range(total_frames)):

    ret, frame = cap.read()

    if not ret:
        break

    detections = detect_faces(frame, face_db)

    for det in detections:

        x, y, w, h = det["bbox"]

        label = det["label"]
        conf = det["confidence"]

        text = f"{label} {conf:.2f}"

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

        cv2.putText(
            frame,
            text,
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

    out.write(frame)

cap.release()
out.release()

print("Video Processing Completed")

  3%|▎         | 22/859 [04:19<2:44:48, 11.81s/it]


KeyboardInterrupt: 

In [ ]:
files.download("output.mp4")